# The chain rule and computation graphs

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The problem
A neural network is a nested function: the output of each layer is the input of the next. To train it we need to know how sensitive the loss is to a weight that sits several layers back. The **chain rule** gives exactly that.

## The rule
If $y=f(u)$ and $u=g(x)$:

$$\frac{dy}{dx}=\frac{dy}{du}\cdot\frac{du}{dx}$$

In words: sensitivities along a path **multiply**. Think of gears: if one turn of gear A turns gear B three times, and one turn of B turns C twice, then one turn of A turns C $3\times2=6$ times.

## Worked example
Split $y=(3x+1)^2$ into two steps: $u=3x+1$ and $y=u^2$.

| Step | Formula | At $x=1$ |
|---|---|---|
| Forward: u | $3x+1$ | 4 |
| Forward: y | $u^2$ | 16 |
| $dy/du$ | $2u$ | 8 |
| $du/dx$ | 3 | 3 |
| $dy/dx$ | $2u\times3$ | 24 |

Check: $y=9x^2+6x+1\Rightarrow y'=18x+6=24$ ✓.

## A computation graph: one neuron
Break a neuron with squared error into simple steps:

$$z=wx+b\qquad a=\sigma(z)\qquad L=(a-y)^2$$

> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-chain

With $x=2,\ w=0.5,\ b=-0.5,\ y=1$:

| Forward pass | Value | Backward pass | Value |
|---|---|---|---|
| $z=wx+b$ | 0.5 | $\partial L/\partial a=2(a-y)$ | −0.755 |
| $a=\sigma(z)$ | 0.622 | $\partial a/\partial z=a(1-a)$ | 0.235 |
| $L=(a-y)^2$ | 0.143 | $\partial L/\partial z=-0.755\times0.235$ | −0.177 |
| | | $\partial L/\partial w=\partial L/\partial z\times x$ | −0.355 |
| | | $\partial L/\partial b=\partial L/\partial z\times1$ | −0.177 |

> Notice that $\partial L/\partial z$ was computed once and **reused** for both w and b. That is backpropagation: the chain rule plus storing intermediate results. It is why the gradient of millions of weights costs about as much as one forward pass.

## Several paths: add them
If a variable affects the output through **several paths**, the contributions add:

$$\frac{\partial L}{\partial x}=\sum_{\text{paths}}\frac{\partial L}{\partial u_k}\frac{\partial u_k}{\partial x}$$

Example: $L=u+v$ with $u=x^2$ and $v=3x$. Then $dL/dx=1\times2x+1\times3=2x+3$.

## Local derivatives of common nodes
| Node | Forward | Backward: gradient to the inputs |
|---|---|---|
| Sum $c=a+b$ | add | Passes the gradient unchanged to both |
| Product $c=ab$ | multiply | Gives $g\cdot b$ to a and $g\cdot a$ to b (swapped) |
| max | the larger one | Only the larger input receives it |
| ReLU | $\max(0,z)$ | Passes if $z>0$, otherwise zero |
| Sigmoid | $\sigma(z)$ | Multiplies by $a(1-a)$ |

## Automatic differentiation (autograd)
Libraries such as PyTorch and TensorFlow do this automatically: they record the graph while computing, then apply the chain rule from the end back to the start. The code below is a 30-line version of that idea:



In [ ]:
import math

class Value:
    """A number that remembers how it was computed, so it can backpropagate."""
    def __init__(self, data, parents=(), name=""):
        self.data, self.grad, self.parents, self.name = data, 0.0, parents, name
        self._backward = lambda: None
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other))
        def back():
            self.grad += out.grad; other.grad += out.grad
        out._backward = back; return out
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other))
        def back():
            self.grad += other.data * out.grad; other.grad += self.data * out.grad
        out._backward = back; return out
    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.data)); out = Value(s, (self,))
        def back():
            self.grad += s * (1 - s) * out.grad
        out._backward = back; return out
    def sq_error(self, y):
        out = Value((self.data - y) ** 2, (self,))
        def back():
            self.grad += 2 * (self.data - y) * out.grad
        out._backward = back; return out
    def backward(self):
        order, seen = [], set()
        def visit(v):
            if v not in seen:
                seen.add(v)
                for p in v.parents: visit(p)
                order.append(v)
        visit(self); self.grad = 1.0
        for v in reversed(order): v._backward()

x, w, b = Value(2.0), Value(0.5), Value(-0.5)
z = w * x + b
a = z.sigmoid()
L = a.sq_error(1.0)
L.backward()
print(f"z = {z.data:.3f}  a = {a.data:.3f}  L = {L.data:.4f}")
print(f"dL/dz = {z.grad:.4f}  dL/dw = {w.grad:.4f}  dL/db = {b.grad:.4f}  dL/dx = {x.grad:.4f}")

# Check dL/dw with a numerical derivative
def loss(wv):
    a = 1 / (1 + math.exp(-(wv * 2.0 - 0.5))); return (a - 1.0) ** 2
h = 1e-6
print("numeric dL/dw =", round((loss(0.5 + h) - loss(0.5 - h)) / (2 * h), 4))

### Exercise

Differentiate $y=\ln(1+e^{2x})$ with the chain rule and evaluate it at $x=0$.

In [ ]:
# Your answer here

### Exercise

Press "One gradient-descent step" in the lab several times. What happens to L, and why?

In [ ]:
# Your answer here